In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import pandas as pd

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
print(device)

cuda


In [5]:
file_path_train = '/content/drive/My Drive/001.PhD/Smartphone/traintest/trainDataset2.csv'

In [6]:
df = pd.read_csv(file_path_train)

In [7]:
df.head()

,sequence
0,"android.net.nsd.STATE_CHANGED, android.net.con..."
1,"android.net.nsd.STATE_CHANGED, android.net.con..."
2,"android.intent.action.DREAMING_STARTED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.intent.action.DREAMING_STARTED, androi..."


In [8]:
sequence_list = df['sequence'].tolist()

In [9]:
# print(sequence_list[0])

In [10]:
sequence_list = sequence_list[:20000]
print(len(sequence_list))

20000


In [11]:
trainDataset = []
for sequence in sequence_list:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    trainDataset.append(result)

In [12]:
# print(trainDataset[0])

In [13]:
# trainDataset[0][0:-1]

In [14]:
from collections import Counter

In [15]:
all_tokens = [token for seq in trainDataset for token in seq]

In [16]:
counter = Counter(all_tokens)

In [17]:
# print(counter)

In [18]:
# Vocabulary
vocab = {token: idx + 1 for idx, token in enumerate(counter.keys())}
vocab["<PAD>"] = 0

In [19]:
inv_vocab = {idx: token for token, idx in vocab.items()}
vocab_size = len(vocab)

In [20]:
print(vocab_size)

62


In [ ]:
from torch.nn.utils.rnn import pad_sequence

In [ ]:
X = []
y = []

In [ ]:
for seq in trainDataset:
    input_seq = [vocab[token] for token in seq[:-1]]
    target = vocab[seq[-1]]

    X.append(torch.tensor(input_seq))
    y.append(target)

In [ ]:
# Pad sequences
X_padded = pad_sequence(X, batch_first=True, padding_value=0)
y = torch.tensor(y)

In [ ]:
import torch.nn as nn

In [ ]:
class VanillaLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        logits = self.fc(hidden[-1])  # last hidden state
        return logits

In [ ]:
model = VanillaLSTM(
    vocab_size=vocab_size,
    embed_dim=64,
    hidden_dim=128
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 230

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X_padded)
    loss = criterion(outputs, y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 4.1556
Epoch 2, Loss: 4.1429
Epoch 3, Loss: 4.1304
Epoch 4, Loss: 4.1179
Epoch 5, Loss: 4.1050
Epoch 6, Loss: 4.0915
Epoch 7, Loss: 4.0768
Epoch 8, Loss: 4.0605
Epoch 9, Loss: 4.0419
Epoch 10, Loss: 4.0200
Epoch 11, Loss: 3.9932
Epoch 12, Loss: 3.9591
Epoch 13, Loss: 3.9143
Epoch 14, Loss: 3.8533
Epoch 15, Loss: 3.7682
Epoch 16, Loss: 3.6488
Epoch 17, Loss: 3.4864
Epoch 18, Loss: 3.2832
Epoch 19, Loss: 3.0624
Epoch 20, Loss: 2.8678
Epoch 21, Loss: 2.7327
Epoch 22, Loss: 2.6528
Epoch 23, Loss: 2.6053
Epoch 24, Loss: 2.5732
Epoch 25, Loss: 2.5501
Epoch 26, Loss: 2.5346
Epoch 27, Loss: 2.5264
Epoch 28, Loss: 2.5239
Epoch 29, Loss: 2.5247
Epoch 30, Loss: 2.5256
Epoch 31, Loss: 2.5240
Epoch 32, Loss: 2.5185
Epoch 33, Loss: 2.5092
Epoch 34, Loss: 2.4970
Epoch 35, Loss: 2.4832
Epoch 36, Loss: 2.4697
Epoch 37, Loss: 2.4582
Epoch 38, Loss: 2.4495
Epoch 39, Loss: 2.4434
Epoch 40, Loss: 2.4392
Epoch 41, Loss: 2.4359
Epoch 42, Loss: 2.4326
Epoch 43, Loss: 2.4291
Epoch 44, Loss: 2.42

In [ ]:
file_path_test = '/content/drive/My Drive/001.PhD/Smartphone/traintest/testDataset2.csv'

In [ ]:
dfTest = pd.read_csv(file_path_test)

In [ ]:
dfTest.head()

,sequence
0,"android.net.wifi.RSSI_CHANGED, android.intent...."
1,"android.intent.action.DREAMING_STARTED, androi..."
2,"android.intent.action.DREAMING_STOPPED, androi..."
3,"android.net.wifi.RSSI_CHANGED, android.intent...."
4,"android.net.wifi.RSSI_CHANGED, android.intent...."


In [ ]:
sequence_list_test = dfTest['sequence'].tolist()

In [ ]:
sequence_list_test = sequence_list_test[:8000]

In [ ]:
testDataset = []
for sequence in sequence_list_test:
    result = sequence.split(",")
    result = [item.strip() for item in result]
    testDataset.append(result)

In [ ]:
X_test = []
y_test = []

In [ ]:
for seq in testDataset:
    # Skip sequences with unseen tokens (optional but safe)
    if any(token not in vocab for token in seq):
        continue

    input_seq = [vocab[token] for token in seq[:-1]]
    target = vocab[seq[-1]]

    X_test.append(torch.tensor(input_seq))
    y_test.append(target)

In [ ]:
X_test_padded = pad_sequence(X_test, batch_first=True, padding_value=0)
y_test = torch.tensor(y_test)

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(X_test_padded)
    y_pred = torch.argmax(logits, dim=1)

In [ ]:
y_true = y_test.cpu().numpy()
y_pred = y_pred.cpu().numpy()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
precision, recall, _, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="micro",
    zero_division=0
)

_, _, f1_macro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

_, _, f1_weighted, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

In [ ]:
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1 (macro)   : {f1_macro:.4f}")
print(f"F1 (weighted): {f1_weighted:.4f}")

Precision    : 0.7494
Recall       : 0.7494
F1 (macro)   : 0.1655
F1 (weighted): 0.7276
